# Notebook 09 — resize comparison on held-out images

Compare the **same saved eight-class ConvNeXt V2 Tiny checkpoint** with two fixed input methods: the existing square resize, and a resize that preserves proportions with mean-colour padding. The primary comparison is **original validation images with horizontal-flip averaging**. Ordinary original predictions and the existing wide/tall stress views are secondary checks. There is no padding-type or weight search.

**Inputs:** upload `09_resize_comparison_inputs.zip` as a private Kaggle dataset and attach it, together with the original **Competition** input. The pack contains the correct eight-class split-trained checkpoint, metadata and old validation predictions. It does not use the two-class specialist or a full-data checkpoint.

**Settings:** enable a GPU. The model weights are already in the input pack; no Hugging Face secret or Internet connection is needed. Keep the standard Kaggle package environment. This notebook installs no packages and performs inference only.

Allow roughly **15–40 minutes**, depending on image loading. Each completed stage writes a small backup containing predictions and results. The model remains in the attached input dataset. Verified completed predictions are reused on another run; attach the saved output/backup to recover them after a new session.

This checkpoint was selected at epoch 7 of its earlier eight-epoch run. It was trained outside the exact 9,547-image holdout. It is one component of the 0.9806 ensemble, **not an independent validation of that complete ensemble**. The split has already been used for development. Changing inference preprocessing can hurt a model trained with square resizing; this is a diagnostic comparison, not a promised improvement. Test images and submission generation are outside this notebook.


In [1]:
from pathlib import Path
from io import BytesIO
import hashlib, json, os, platform, shutil, zipfile
import numpy as np
import pandas as pd
from PIL import Image
import PIL
import torch
import torchvision
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from torch.torch_version import TorchVersion
import timm
from tqdm.auto import tqdm
from IPython.display import display

INPUT_PATH = None
BASE = None
RECOVERY_SOURCE = None
OUT = Path('/kaggle/working/09_resize_comparison')
MODEL_NAME = 'convnextv2_tiny.fcmae_ft_in1k'
CLASSES = ['CatEye','CatFace','CatMouth','CatNose','DogEye','DogFace','DogMouth','DogNose']
SIZE, BATCH_SIZE, NUM_WORKERS = 192, 64, 0
MEAN, STD = [.485,.456,.406], [.229,.224,.225]
FILL = tuple(round(255 * x) for x in MEAN)
EXPECTED_MANIFEST_SHA = 'ecfba4a0b35a2fa91909fb0282b477b5332645e0a96c3e8e59c2100626c4bf3c'
EXPECTED_SPLIT_HASH = '842af9ee7beb6fe8c85fd55163deeb15cf8a5c778506c075f987e9b9ed9889b6'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before continuing.'
device = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.set_num_threads(2)
versions = dict(python=platform.python_version(), torch=str(torch.__version__),
                torchvision=str(torchvision.__version__), timm=str(timm.__version__),
                numpy=str(np.__version__), pandas=str(pd.__version__), pillow=str(PIL.__version__))
print('GPU:', torch.cuda.get_device_name(0), '| versions:', versions, flush=True)
print('Saved eight-class V2; validation inference only. Padding colour:', FILL, flush=True)


GPU: Tesla T4 | versions: {'python': '3.13.15', 'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128', 'timm': '1.0.29', 'numpy': '2.1.3', 'pandas': '2.3.3', 'pillow': '13.0.0'}
Saved eight-class V2; validation inference only. Padding colour: (124, 116, 104)


## 1 Verify the input pack and holdout

Check the checkpoint fingerprint, original labels, class order and train/validation separation before loading the model. A small fixed image check verifies that Competition is the expected dataset.


In [2]:
def sha_bytes(raw):
    return hashlib.sha256(raw).hexdigest()

def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(4*1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def atomic_json(path, value):
    path = Path(path); temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(value, indent=2) + '\n'); os.replace(temp, path)

def read_pack(path):
    path = Path(path); name = '09_resize_comparison_manifest.json'
    if path.is_dir():
        raw = (path/name).read_bytes(); record = json.loads(raw)
        data = {key: (path/key).read_bytes() for key in record['files']}
    else:
        with zipfile.ZipFile(path) as z:
            assert z.testzip() is None, 'Input ZIP failed its CRC check.'
            assert len(z.namelist()) == len(set(z.namelist()))
            raw = z.read(name); record = json.loads(raw)
            data = {key: z.read(key) for key in record['files']}
    assert sha_bytes(raw) == EXPECTED_MANIFEST_SHA, 'Wrong or changed Notebook 09 input pack.'
    for key, info in record['files'].items():
        assert sha_bytes(data[key]) == info['sha256'] and len(data[key]) == info['bytes'], key
    return record, data

def image_folder(base):
    for path in [base/'train'/'train', base/'train']:
        if (path/'00001.jpg').is_file(): return path
    raise FileNotFoundError(f'Cannot find training images under {base}')

def check_probabilities(p, n):
    assert p.shape == (n, len(CLASSES)) and np.isfinite(p).all()
    assert (p >= 0).all() and (p <= 1).all()
    assert np.allclose(p.sum(1), 1, atol=1e-5, rtol=0), 'Invalid probability rows.'

def read_reference(raw, ids, truth):
    with np.load(BytesIO(raw), allow_pickle=False) as a:
        assert a['classes'].tolist() == CLASSES
        assert np.array_equal(a['image_id'], ids) and np.array_equal(a['labels'], truth)
        assert str(a['split_hash']) == EXPECTED_SPLIT_HASH
        p = a['probs'].copy()
    check_probabilities(p, len(ids)); return p

def validate_frames(data, labels):
    meta = pd.read_csv(BytesIO(data['train_metadata_clean.csv']))
    tr = pd.read_csv(BytesIO(data['train_split_1337.csv']))
    va = pd.read_csv(BytesIO(data['val_split_1337.csv']))
    assert len(meta) == 63645 and len(tr) == 54098 and len(va) == 9547
    assert meta.image_id.is_unique and meta.md5.is_unique and tr.image_id.is_unique and va.image_id.is_unique
    assert set(tr.image_id).isdisjoint(set(va.image_id))
    assert set(tr.image_id) | set(va.image_id) == set(meta.image_id)
    official = labels.set_index('image_id')['class']; indexed = meta.set_index('image_id')
    assert np.array_equal(meta['class'], official.loc[meta.image_id])
    for frame in [tr, va]:
        assert np.array_equal(frame['class'], official.loc[frame.image_id])
    assert set(indexed.loc[tr.image_id, 'md5']).isdisjoint(set(indexed.loc[va.image_id, 'md5']))
    assert sha_bytes(va.image_id.to_numpy(dtype='<i8').tobytes()) == EXPECTED_SPLIT_HASH
    frame = indexed.loc[va.image_id].reset_index()
    frame['label'] = frame['class'].map({c:i for i,c in enumerate(CLASSES)}).astype('int64')
    return meta, tr, frame

if INPUT_PATH is None:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('09_resize_comparison_manifest.json')]
    if not candidates: candidates = list(Path('/kaggle/input').rglob('09_resize_comparison_inputs.zip'))
    assert len(candidates) == 1, f'Attach the Notebook 09 private input pack, or set INPUT_PATH. Found: {candidates}'
    INPUT_PATH = candidates[0]
pack, pack_files = read_pack(INPUT_PATH)
assert pack['classes'] == CLASSES and pack['split_hash'] == EXPECTED_SPLIT_HASH
SOURCE_CFG = json.loads(pack_files['v2_config.json'])
CHECKPOINT_SHA = sha_bytes(pack_files['v2_best.pth'])
assert CHECKPOINT_SHA == pack['checkpoint_sha256']
assert SOURCE_CFG['model'] == MODEL_NAME and SOURCE_CFG['classes'] == CLASSES
assert not SOURCE_CFG['full_data'] and SOURCE_CFG['split_seed'] == 1337 and SOURCE_CFG['train_seed'] == 42
assert SOURCE_CFG['split_hash'] == EXPECTED_SPLIT_HASH
assert SOURCE_CFG['mean'] == MEAN and SOURCE_CFG['std'] == STD and SOURCE_CFG['size'] == SIZE
if BASE is None:
    candidates = []
    for path in Path('/kaggle/input').rglob('train_labels.csv'):
        try: image_folder(path.parent)
        except FileNotFoundError: continue
        candidates.append(path.parent)
    assert len(candidates) == 1, f'Attach Competition, or set BASE. Found: {candidates}'
    BASE = candidates[0]
BASE = Path(BASE); TRAIN_DIR = image_folder(BASE)
assert sha_file(BASE/'train_labels.csv') == pack['label_sha256'], 'Competition labels differ.'
labels = pd.read_csv(BASE/'train_labels.csv')
assert labels.columns.tolist() == ['image_id','class'] and len(labels) == 63660
meta, train_fold, val_df = validate_frames(pack_files, labels)
val_ids = val_df.image_id.to_numpy(dtype='int64'); y = val_df.label.to_numpy(dtype='int64')
references = {v: read_reference(pack_files[f'reference_val_{v}.npz'], val_ids, y) for v in ['clean','tta','wide','tall']}
for view, p in references.items():
    assert int((p.argmax(1) != y).sum()) == pack['baseline_errors'][view]
for r in val_df.sample(n=32, random_state=42).itertuples(index=False):
    assert hashlib.md5((TRAIN_DIR/f'{int(r.image_id):05d}.jpg').read_bytes()).hexdigest() == r.md5
EVAL_CFG = dict(experiment='09-fixed-resize-comparison-v1', source_manifest_sha256=EXPECTED_MANIFEST_SHA,
                checkpoint_sha256=CHECKPOINT_SHA, split_hash=EXPECTED_SPLIT_HASH, classes=CLASSES,
                size=SIZE, methods=['stretch','letterbox'], padding='constant ImageNet mean RGB', fill=list(FILL),
                resize_interpolation='Pillow bilinear', letterbox_rounding='round, clamp to [1,size], extra padding right/bottom',
                primary=dict(case='original', averaging='tta'),
                secondary=dict(original_ordinary=True, cases=['legacy_wide','legacy_tall'], averaging='ordinary'),
                legacy_stress='square resize first, then reflect-pad round(size*.15) on vertical (wide) or horizontal (tall) sides',
                mean=MEAN, std=STD, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, amp='float16',
                source_code_sha256='984bcb143444646b6cc18e461b7fcf33a5362ef6938e1a7e75a4c04f66716ec1', versions=versions,
                gpu=torch.cuda.get_device_name(0), cudnn_deterministic=True,
                cuda_matmul_tf32=bool(torch.backends.cuda.matmul.allow_tf32), cudnn_tf32=bool(torch.backends.cudnn.allow_tf32))
EVAL_SHA = sha_bytes(json.dumps(EVAL_CFG, sort_keys=True).encode())
OUT.mkdir(parents=True, exist_ok=True)
plan_path = OUT/'evaluation_plan.json'
if plan_path.exists():
    assert json.loads(plan_path.read_text()) == dict(config_sha256=EVAL_SHA, config=EVAL_CFG), 'Existing outputs use different settings; preserve them before starting a different run.'
else:
    assert not list(OUT.glob('*.npz')), 'Prediction files exist without their evaluation plan; preserve and review them.'
    atomic_json(plan_path, dict(config_sha256=EVAL_SHA, config=EVAL_CFG))
atomic_json(OUT/'source_manifest.json', pack)
(OUT/'source_model_config.json').write_bytes(pack_files['v2_config.json'])
val_df.to_csv(OUT/'validation_metadata.csv', index=False)
print('Verified: 54,098 training IDs separated from 9,547 held-out IDs.', flush=True)
print('Correct eight-class checkpoint:', CHECKPOINT_SHA, '| selected epoch:', pack['selected_epoch'], flush=True)
print('Fixed comparison:', EVAL_SHA, flush=True)


Verified: 54,098 training IDs separated from 9,547 held-out IDs.
Correct eight-class checkpoint: b2162e667cc1cb304b4ea8b57816253ad67f8c825a1cd84d38050cfce7973e1c | selected epoch: 7
Fixed comparison: 96d668d1812b39925e95ad04df71bc59faa4d5a08f63c120f63cff5b407a5879


## 2 Load the saved model and the paired comparison

The letterbox method resizes the complete image to fit inside 192×192, then centres it on a fixed RGB (124,116,104) background. The stretch method uses the original bilinear square resize. The same decoded image, checkpoint, normalization, labels and flip averaging are used for both methods.

The secondary stress views reproduce the earlier protocol: resize to a square, add reflected borders to one axis, then apply each final resize method. These are artificial diagnostics, not claims about the exact transformation used to create test images. The original-image comparison is primary.


In [3]:
def make_view(image, case, size):
    if case == 'original': return image.copy()
    assert case in ['legacy_wide','legacy_tall']
    square = image.resize((size,size), Image.Resampling.BILINEAR)
    pad = round(size*.15)
    widths = ((pad,pad),(0,0),(0,0)) if case == 'legacy_wide' else ((0,0),(pad,pad),(0,0))
    return Image.fromarray(np.pad(np.asarray(square), widths, mode='reflect'))

def resize_image(image, method, size, fill):
    if method == 'stretch': return image.resize((size,size), Image.Resampling.BILINEAR)
    assert method == 'letterbox'
    width, height = image.size
    scale = size/max(width,height)
    resized = (max(1,min(size,round(width*scale))), max(1,min(size,round(height*scale))))
    small = image.resize(resized, Image.Resampling.BILINEAR)
    canvas = Image.new('RGB', (size,size), fill)
    canvas.paste(small, ((size-resized[0])//2, (size-resized[1])//2))
    return canvas

class PairedImages(Dataset):
    def __init__(self, frame, folder, case, methods):
        self.frame = frame.reset_index(drop=True); self.folder = folder
        self.case, self.methods = case, methods
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        r = self.frame.iloc[i]; image_id = int(r.image_id)
        with Image.open(self.folder/f'{image_id:05d}.jpg') as image:
            source = make_view(image.convert('RGB'), self.case, SIZE)
            tensors = {method: TF.normalize(TF.to_tensor(resize_image(source,method,SIZE,FILL)), MEAN, STD)
                       for method in self.methods}
        return tensors, int(r.label), image_id

def prediction_keys(case):
    assert case in ['original','legacy_wide','legacy_tall']
    return [(method,avg) for method in ['stretch','letterbox']
            for avg in (['ordinary','tta'] if case == 'original' else ['ordinary'])]

def prediction_name(case, method, averaging):
    assert (method,averaging) in prediction_keys(case)
    return f'{case}_{method}_{averaging}.npz'

def checked_prediction(raw, case, method, averaging):
    with np.load(BytesIO(raw), allow_pickle=False) as a:
        assert a['classes'].tolist() == CLASSES and np.array_equal(a['image_id'], val_ids)
        assert np.array_equal(a['labels'], y)
        for key, value in dict(config_sha256=EVAL_SHA, checkpoint_sha256=CHECKPOINT_SHA,
                               source_manifest_sha256=EXPECTED_MANIFEST_SHA, split_hash=EXPECTED_SPLIT_HASH,
                               case=case, method=method, averaging=averaging).items():
            assert str(a[key]) == value, f'Wrong prediction metadata: {key}'
        p = a['probs'].copy()
    check_probabilities(p, len(val_ids)); return p

def cache_manifest():
    path = OUT/'prediction_manifest.json'
    if not path.exists(): return dict(config_sha256=EVAL_SHA, checkpoint_sha256=CHECKPOINT_SHA, files={})
    record = json.loads(path.read_text())
    assert record['config_sha256'] == EVAL_SHA and record['checkpoint_sha256'] == CHECKPOINT_SHA
    return record

def load_prediction(case, method, averaging):
    name = prediction_name(case,method,averaging); path = OUT/name
    raw = path.read_bytes(); record = cache_manifest()
    if name in record['files']: assert sha_bytes(raw) == record['files'][name], f'Changed prediction: {name}'
    return checked_prediction(raw,case,method,averaging)

def save_prediction(p, case, method, averaging):
    check_probabilities(p, len(val_ids)); name = prediction_name(case,method,averaging)
    path = OUT/name; temp = path.with_name(path.name+'.tmp')
    data = dict(probs=p.astype('float32'), image_id=val_ids, labels=y, classes=np.array(CLASSES),
                config_sha256=np.array(EVAL_SHA), checkpoint_sha256=np.array(CHECKPOINT_SHA),
                source_manifest_sha256=np.array(EXPECTED_MANIFEST_SHA), split_hash=np.array(EXPECTED_SPLIT_HASH),
                case=np.array(case), method=np.array(method), averaging=np.array(averaging))
    with temp.open('wb') as f: np.savez_compressed(f, **data)
    os.replace(temp,path)
    checked_prediction(path.read_bytes(),case,method,averaging)
    record = cache_manifest(); record['files'][name] = sha_file(path)
    atomic_json(OUT/'prediction_manifest.json',record)

def recover_outputs():
    source = RECOVERY_SOURCE
    if source is None:
        candidates = []
        for path in Path('/kaggle/input').rglob('evaluation_plan.json'):
            try: record = json.loads(path.read_text())
            except (ValueError,OSError): continue
            if record.get('config_sha256') == EVAL_SHA: candidates.append(path.parent)
        if not candidates: candidates = list(Path('/kaggle/input').rglob('09_resize_comparison_backup.zip'))
        assert len(candidates) <= 1, f'Multiple recovery sources; set RECOVERY_SOURCE explicitly: {candidates}'
        source = candidates[0] if candidates else None
    if source is None: return
    source = Path(source)
    allowed = [prediction_name(c,m,a) for c in ['original','legacy_wide','legacy_tall'] for m,a in prediction_keys(c)]
    if source.is_dir():
        read = lambda name: (source/name).read_bytes()
        present = {name for name in allowed if (source/name).is_file()}
        plan = json.loads(read('evaluation_plan.json')); record = json.loads(read('prediction_manifest.json'))
        blobs = {name: read(name) for name in present}
    else:
        with zipfile.ZipFile(source) as z:
            assert z.testzip() is None
            prefix = '09_resize_comparison/'
            plan = json.loads(z.read(prefix+'evaluation_plan.json'))
            record = json.loads(z.read(prefix+'prediction_manifest.json'))
            blobs = {name: z.read(prefix+name) for name in allowed if prefix+name in z.namelist()}
    assert plan == dict(config_sha256=EVAL_SHA,config=EVAL_CFG), 'Recovery settings/environment do not match.'
    assert record['config_sha256'] == EVAL_SHA and record['checkpoint_sha256'] == CHECKPOINT_SHA
    current = cache_manifest(); restored = 0
    for case in ['original','legacy_wide','legacy_tall']:
        for method,avg in prediction_keys(case):
            name = prediction_name(case,method,avg)
            if name not in blobs or (OUT/name).exists(): continue
            assert sha_bytes(blobs[name]) == record['files'][name]
            checked_prediction(blobs[name],case,method,avg)
            temp = OUT/(name+'.tmp'); temp.write_bytes(blobs[name]); os.replace(temp,OUT/name)
            current['files'][name] = sha_bytes(blobs[name]); restored += 1
    atomic_json(OUT/'prediction_manifest.json',current)
    print('Recovered verified prediction files:',restored,flush=True)

def metrics(truth, p):
    check_probabilities(p,len(truth))
    return dict(errors=int((p.argmax(1)!=truth).sum()), accuracy=float((p.argmax(1)==truth).mean()),
                nll=float(-np.log(p.astype('float64')[np.arange(len(truth)),truth].clip(1e-12,1)).mean()))

def paired_row(case, averaging, before, after, truth, group='all_validation'):
    old,new = before.argmax(1),after.argmax(1)
    b,a = metrics(truth,before),metrics(truth,after)
    return dict(case=case,averaging=averaging,group=group,images=len(truth),
                stretch_errors=b['errors'],letterbox_errors=a['errors'],
                stretch_accuracy=b['accuracy'],letterbox_accuracy=a['accuracy'],
                fixed=int(((old!=truth)&(new==truth)).sum()),broken=int(((old==truth)&(new!=truth)).sum()),
                changed=int((old!=new).sum()),stretch_nll=b['nll'],letterbox_nll=a['nll'])

def write_summary():
    rows=[]; drift=[]; completed=[]
    for case in ['original','legacy_wide','legacy_tall']:
        if not all((OUT/prediction_name(case,m,a)).is_file() for m,a in prediction_keys(case)): continue
        completed.append(case)
        for avg in (['ordinary','tta'] if case=='original' else ['ordinary']):
            before=load_prediction(case,'stretch',avg);after=load_prediction(case,'letterbox',avg)
            rows.append(paired_row(case,avg,before,after,y))
            ref=references[('tta' if avg=='tta' else 'clean') if case=='original' else case.replace('legacy_','')]
            drift.append(dict(case=case,averaging=avg,saved_errors=metrics(y,ref)['errors'],
                              fresh_errors=metrics(y,before)['errors'],
                              argmax_disagreements=int((ref.argmax(1)!=before.argmax(1)).sum()),
                              mean_abs_probability_difference=float(np.abs(ref-before).mean()),
                              max_abs_probability_difference=float(np.abs(ref-before).max())))
    table=pd.DataFrame(rows);table.to_csv(OUT/'resize_comparison.csv',index=False)
    pd.DataFrame(drift).to_csv(OUT/'baseline_reproduction.csv',index=False)
    if 'original' in completed:
        before=load_prediction('original','stretch','tta');after=load_prediction('original','letterbox','tta')
        short=val_df[['width','height']].min(axis=1).to_numpy();square=np.abs(val_df.aspect.to_numpy()-1)<.05
        groups={'near_square':square,'non_square':~square,'short_side_under32':short<32,
                'short_side32plus':short>=32,'true_cat_pair':np.isin(y,[2,3])}
        groups.update({c:y==i for i,c in enumerate(CLASSES)})
        subgroup=[paired_row('original','tta',before[mask],after[mask],y[mask],name)
                  for name,mask in groups.items() if mask.any()]
        pd.DataFrame(subgroup).to_csv(OUT/'original_tta_subgroups.csv',index=False)
        old,new=before.argmax(1),after.argmax(1)
        changes=val_df[['image_id','class','width','height']].copy()
        changes['stretch_class']=np.array(CLASSES)[old];changes['letterbox_class']=np.array(CLASSES)[new]
        changes['fixed']=(old!=y)&(new==y);changes['broken']=(old==y)&(new!=y)
        changes.loc[old!=new].to_csv(OUT/'original_tta_changed_predictions.csv',index=False)
        for name,pred in [('stretch',old),('letterbox',new)]:
            matrix=np.zeros((len(CLASSES),len(CLASSES)),dtype='int64');np.add.at(matrix,(y,pred),1)
            pd.DataFrame(matrix,index=CLASSES,columns=CLASSES).to_csv(OUT/f'original_tta_confusion_{name}.csv')
    atomic_json(OUT/'progress.json',dict(config_sha256=EVAL_SHA,checkpoint_sha256=CHECKPOINT_SHA,
                completed_cases=completed,complete=len(completed)==3,primary_case='original',primary_averaging='tta',
                gpu_training_performed=False,test_inference_performed=False,submission_created=False,
                independent_validation_of_full_winner=False))
    return table,pd.DataFrame(drift)

def export_backup():
    target=OUT.parent/'09_resize_comparison_backup.zip';temp=target.with_name(target.name+'.tmp')
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for path in sorted(OUT.rglob('*')):
            if path.is_file() and not path.name.endswith('.tmp'): z.write(path,path.relative_to(OUT.parent))
    with zipfile.ZipFile(temp) as z: assert z.testzip() is None, 'Output backup failed its CRC check.'
    os.replace(temp,target);print('Verified backup:',target,flush=True)
    return target

@torch.inference_mode()
def predict_pair(case, methods):
    model.eval()
    loader=DataLoader(PairedImages(val_df,TRAIN_DIR,case,methods),batch_size=BATCH_SIZE,shuffle=False,
                      num_workers=NUM_WORKERS,pin_memory=True,drop_last=False)
    collected={(m,a):[] for m in methods for a in (['ordinary','tta'] if case=='original' else ['ordinary'])}
    labels_out=[];ids=[]
    for tensors,truth,image_ids in tqdm(loader,desc=f'{case}: paired resize comparison',leave=True):
        labels_out.append(truth.numpy());ids.append(image_ids.numpy())
        for method in methods:
            x=tensors[method].to(device,non_blocking=True)
            with torch.autocast('cuda',dtype=torch.float16): logits=model(x)
            p=logits.float().softmax(1)
            collected[(method,'ordinary')].append(p.cpu().numpy())
            if case=='original':
                with torch.autocast('cuda',dtype=torch.float16): flipped=model(torch.flip(x,[3]))
                averaged=(p+flipped.float().softmax(1))/2
                collected[(method,'tta')].append(averaged.cpu().numpy())
            del x
    assert np.array_equal(np.concatenate(ids),val_ids) and np.array_equal(np.concatenate(labels_out),y)
    del loader
    return {key:np.concatenate(parts) for key,parts in collected.items()}

def run_case(case):
    keys=prediction_keys(case)
    found={(m,a):load_prediction(case,m,a) for m,a in keys if (OUT/prediction_name(case,m,a)).exists()}
    if len(found)==len(keys): print('Reused verified completed predictions:',case,flush=True)
    else:
        methods=[m for m in ['stretch','letterbox'] if any((m,a) not in found for mm,a in keys if mm==m)]
        fresh=predict_pair(case,methods)
        for m,a in keys:
            if (m,a) not in found: save_prediction(fresh[(m,a)],case,m,a)
    table,drift=write_summary();export_backup()
    display(table);display(drift)
    if not drift.empty and (drift.argmax_disagreements>5).any():
        print('Fresh baseline differs from the older saved predictions; review the reproduction table before interpreting small changes.',flush=True)
    return table

recover_outputs()
# The older trusted checkpoint includes a TorchVersion string subclass.
# Allow only that named PyTorch type in the restricted weights loader.
with torch.serialization.safe_globals([TorchVersion]):
    checkpoint=torch.load(BytesIO(pack_files['v2_best.pth']),map_location='cpu',weights_only=True)
assert checkpoint['epoch']==pack['selected_epoch']
for key in ['model','size','classes','split_seed','train_seed','full_data','split_hash','mean','std']:
    assert checkpoint['config'][key]==SOURCE_CFG[key], f'Checkpoint configuration mismatch: {key}'
assert checkpoint['model']['head.fc.weight'].shape[0]==len(CLASSES)
model=timm.create_model(MODEL_NAME,pretrained=False,num_classes=len(CLASSES))
model.load_state_dict(checkpoint['model'],strict=True)
del checkpoint
model=model.to(device).eval();model.requires_grad_(False)
print('Saved eight-class checkpoint loaded. Ready for the original-image comparison.',flush=True)


Saved eight-class checkpoint loaded. Ready for the original-image comparison.


## 3 Run the primary comparison

Use all 9,547 original holdout images. Both methods produce ordinary and flip-averaged predictions. The table counts corrected predictions and new mistakes. The baseline-reproduction table compares fresh stretch predictions with the old saved baseline; numerical differences are reported rather than hidden.


In [4]:
primary_table=run_case('original')
print('Primary comparison and backup saved. Review the original TTA row before the stress checks.',flush=True)


original: paired resize comparison:   0%|          | 0/150 [00:00<?, ?it/s]

Verified backup: /kaggle/working/09_resize_comparison_backup.zip


,case,averaging,group,images,stretch_errors,letterbox_errors,stretch_accuracy,letterbox_accuracy,fixed,broken,changed,stretch_nll,letterbox_nll
0,original,ordinary,all_validation,9547,34,161,0.996439,0.983136,1,128,130,0.102212,0.142166
1,original,tta,all_validation,9547,29,154,0.996962,0.983869,1,126,128,0.100978,0.138793


,case,averaging,saved_errors,fresh_errors,argmax_disagreements,mean_abs_probability_difference,max_abs_probability_difference
0,original,ordinary,34,34,0,2.250621e-07,0.000172
1,original,tta,29,29,0,1.913501e-07,0.000098


Primary comparison and backup saved. Review the original TTA row before the stress checks.


## 4 Run the two existing stress checks

These secondary checks use ordinary predictions for both resize methods, matching the old wide/tall baseline. Previously completed cases are reused.


In [5]:
for stress_case in ['legacy_wide','legacy_tall']:
    run_case(stress_case)


legacy_wide: paired resize comparison:   0%|          | 0/150 [00:00<?, ?it/s]

Verified backup: /kaggle/working/09_resize_comparison_backup.zip


,case,averaging,group,images,stretch_errors,letterbox_errors,stretch_accuracy,letterbox_accuracy,fixed,broken,changed,stretch_nll,letterbox_nll
0,original,ordinary,all_validation,9547,34,161,0.996439,0.983136,1,128,130,0.102212,0.142166
1,original,tta,all_validation,9547,29,154,0.996962,0.983869,1,126,128,0.100978,0.138793
2,legacy_wide,ordinary,all_validation,9547,63,79,0.993401,0.991725,35,51,89,0.110690,0.113758


,case,averaging,saved_errors,fresh_errors,argmax_disagreements,mean_abs_probability_difference,max_abs_probability_difference
0,original,ordinary,34,34,0,2.250621e-07,0.000172
1,original,tta,29,29,0,1.913501e-07,0.000098
2,legacy_wide,ordinary,63,63,0,2.632803e-07,0.000177


legacy_tall: paired resize comparison:   0%|          | 0/150 [00:00<?, ?it/s]

Verified backup: /kaggle/working/09_resize_comparison_backup.zip


,case,averaging,group,images,stretch_errors,letterbox_errors,stretch_accuracy,letterbox_accuracy,fixed,broken,changed,stretch_nll,letterbox_nll
0,original,ordinary,all_validation,9547,34,161,0.996439,0.983136,1,128,130,0.102212,0.142166
1,original,tta,all_validation,9547,29,154,0.996962,0.983869,1,126,128,0.100978,0.138793
2,legacy_wide,ordinary,all_validation,9547,63,79,0.993401,0.991725,35,51,89,0.110690,0.113758
3,legacy_tall,ordinary,all_validation,9547,73,66,0.992354,0.993087,48,41,92,0.111910,0.111222


,case,averaging,saved_errors,fresh_errors,argmax_disagreements,mean_abs_probability_difference,max_abs_probability_difference
0,original,ordinary,34,34,0,2.250621e-07,0.000172
1,original,tta,29,29,0,1.913501e-07,0.000098
2,legacy_wide,ordinary,63,63,0,2.632803e-07,0.000177
3,legacy_tall,ordinary,73,73,0,2.026594e-07,0.000183


## 5 Save the completed evaluation

Download the backup ZIP and this executed notebook. Keep the current 0.9806 submission while the results are reviewed. The backup contains all prediction arrays, their fingerprints, comparisons, subgroup results and changed validation predictions. It is small because the checkpoint stays in the input pack.


In [6]:
comparison,reproduction=write_summary()
assert all((OUT/prediction_name(c,m,a)).is_file() for c in ['original','legacy_wide','legacy_tall'] for m,a in prediction_keys(c))
archive=export_backup()
display(comparison)
print('COMPLETE. Download:',archive,flush=True)
print('No model was trained. No test predictions or submission CSV were generated.',flush=True)


Verified backup: /kaggle/working/09_resize_comparison_backup.zip


,case,averaging,group,images,stretch_errors,letterbox_errors,stretch_accuracy,letterbox_accuracy,fixed,broken,changed,stretch_nll,letterbox_nll
0,original,ordinary,all_validation,9547,34,161,0.996439,0.983136,1,128,130,0.102212,0.142166
1,original,tta,all_validation,9547,29,154,0.996962,0.983869,1,126,128,0.100978,0.138793
2,legacy_wide,ordinary,all_validation,9547,63,79,0.993401,0.991725,35,51,89,0.110690,0.113758
3,legacy_tall,ordinary,all_validation,9547,73,66,0.992354,0.993087,48,41,92,0.111910,0.111222


COMPLETE. Download: /kaggle/working/09_resize_comparison_backup.zip
No model was trained. No test predictions or submission CSV were generated.
